In [1]:
import os

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, from_unixtime, to_date
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    LongType,
    StructField,
    StructType,
)

In [2]:
SENSOR_SCHEMA = StructType(
    [
        StructField("farm_sensor_id", IntegerType(), True),
        StructField("farm_id", IntegerType(), True),
        StructField("sensor_type_id", IntegerType(), True),
        StructField("value", DoubleType(), True),
        StructField("timestamp", LongType(), True),
    ]
)

In [3]:
def get_config():
    return {
        # Kafka
        "kafka_bootstrap_servers": os.getenv(
            "KAFKA_BOOTSTRAP_SERVERS",
            "urbangreen-kafka:9092",
        ),
        "kafka_topic": os.getenv(
            "KAFKA_TOPIC_SENSOR_READINGS",
            "sensor_readings",
        ),
        "starting_offsets": os.getenv(
            "KAFKA_STARTING_OFFSETS",
            "earliest",
        ),

        # MinIO
        "minio_endpoint": os.getenv(
            "MINIO_ENDPOINT",
            "http://urbangreen-minio:9000",
        ),
        "minio_access_key": os.getenv(
            "MINIO_ROOT_USER",
            "minioadmin",
        ),
        "minio_secret_key": os.getenv(
            "MINIO_ROOT_PASSWORD",
            "minioadmin123",
        ),
        "staging_bucket": os.getenv(
            "MINIO_STAGING_BUCKET",
            "staging",
        ),

        # Streaming
        "trigger_interval": os.getenv(
            "STREAM_TRIGGER_INTERVAL",
            "60 seconds",
        ),
    }

In [4]:
config = get_config()
config

{'kafka_bootstrap_servers': 'urbangreen-kafka:9092',
 'kafka_topic': 'sensor_readings',
 'starting_offsets': 'earliest',
 'minio_endpoint': 'http://urbangreen-minio:9000',
 'minio_access_key': 'minioadmin',
 'minio_secret_key': 'minioadmin123',
 'staging_bucket': 'staging',
 'trigger_interval': '60 seconds'}

In [5]:
def create_spark_session(config):
    return (
        SparkSession.builder
        .master("spark://urbangreen-spark-master:7077")
        .appName("sensor_readings_stream")

        # Docker -> Jupyter container communication
        .config("spark.driver.host", "urbangreen-jupyter")
        .config("spark.driver.bindAddress", "0.0.0.0")

        # Keep it small for your local cluster (2 workers x 1 core)
        .config("spark.executor.cores", "1")
        .config("spark.executor.memory", "1g")

        # Kafka connector
        .config(
            "spark.jars.packages",
            "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.2,"
            "org.apache.hadoop:hadoop-aws:3.4.1,"
            "software.amazon.awssdk:bundle:2.29.52",
        )

        # MinIO / S3A
        .config(
            "spark.hadoop.fs.s3a.endpoint",
            config["minio_endpoint"],
        )
        .config(
            "spark.hadoop.fs.s3a.access.key",
            config["minio_access_key"],
        )
        .config(
            "spark.hadoop.fs.s3a.secret.key",
            config["minio_secret_key"],
        )
        .config(
            "spark.hadoop.fs.s3a.path.style.access",
            "true",
        )
        .config(
            "spark.hadoop.fs.s3a.connection.ssl.enabled",
            "false",
        )
        .config(
            "spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider",
        )

        # Acceptance criteria from ticket
        .config(
            "spark.sql.streaming.schemaInference",
            "false",
        )

        .getOrCreate()
    )

In [6]:
spark = create_spark_session(config)
spark.sparkContext.setLogLevel("WARN")

:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/jovyan/.ivy2.5.2/cache
The jars for the packages stored in: /home/jovyan/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
software.amazon.awssdk#bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f70648db-6731-485f-9355-68dd6a531c3c;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.0.2 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.0.2 in central
	found org.apache.kafka#kafka-clients;3.9.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.7 in central
	found org.slf4j#slf4j-api;2.0.16 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.1 in central
	found org.apache.hadoop#hado

In [7]:
def read_kafka_stream(spark, config):
    return (
        spark.readStream
        .format("kafka")
        .option(
            "kafka.bootstrap.servers",
            config["kafka_bootstrap_servers"],
        )
        .option("subscribe", config["kafka_topic"])
        .option("startingOffsets", config["starting_offsets"])
        .option("failOnDataLoss", "false")
        .load()
    )

In [8]:
kafka_df = read_kafka_stream(spark, config)
print(kafka_df)

DataFrame[key: binary, value: binary, topic: string, partition: int, offset: bigint, timestamp: timestamp, timestampType: int]


In [9]:
query = (
    kafka_df
    .selectExpr(
        "CAST(key AS STRING)",
        "CAST(value AS STRING)",
        "topic",
        "partition",
        "offset"
    )
    .writeStream
    .format("console")
    .option("truncate", "false")
    .option("numRows", 20)
    .option(
        "checkpointLocation",
        "/tmp/test-kafka-checkpoint"
    )
    .start()
)

query.processAllAvailable()
query.stop()

26/07/09 11:56:21 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+---+-----------------------------------------------------------------------------------------------------+---------------+---------+------+
|key|value                                                                                                |topic          |partition|offset|
+---+-----------------------------------------------------------------------------------------------------+---------------+---------+------+
|2  |{"farm_sensor_id": 2, "farm_id": 1, "sensor_type_id": 2, "value": 59.762, "timestamp": 1752061984}   |sensor_readings|2        |0     |
|3  |{"farm_sensor_id": 3, "farm_id": 1, "sensor_type_id": 3, "value": 518.72, "timestamp": 1752061984}   |sensor_readings|2        |1     |
|9  |{"farm_sensor_id": 9, "farm_id": 2, "sensor_type_id": 3, "value": 515.393, "timestamp": 1752061984}  |sensor_readings|2        |2     |
|16 |{"farm_sensor_id": 16, "farm_id": 3, "sensor_type_id

26/07/09 11:56:32 WARN DAGScheduler: Failed to cancel job group 050115cb-7240-45f0-aaf7-fd17318736ce. Cannot find active jobs for it.
26/07/09 11:56:32 WARN DAGScheduler: Failed to cancel job group 050115cb-7240-45f0-aaf7-fd17318736ce. Cannot find active jobs for it.


In [10]:
def parse_sensor_readings(df):
    return (
        df.select(
            from_json(
                col("value").cast("string"),
                SENSOR_SCHEMA,
            ).alias("data")
        )
        .select("data.*")
        .withColumn(
            "event_date",
            to_date(from_unixtime(col("timestamp"))),
        )
    )

In [11]:
parsed_df = parse_sensor_readings(kafka_df)
print(parsed_df)

DataFrame[farm_sensor_id: int, farm_id: int, sensor_type_id: int, value: double, timestamp: bigint, event_date: date]


In [12]:
query = (
    parsed_df
    .writeStream
    .format("console")
    .option("truncate", "false")
    .option("numRows", 10)
    .option(
        "checkpointLocation",
        "/tmp/test-parsed-checkpoint"
    )
    .start()
)

query.processAllAvailable()
query.stop()

26/07/09 11:56:32 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+--------------+-------+--------------+-------+----------+----------+
|farm_sensor_id|farm_id|sensor_type_id|value  |timestamp |event_date|
+--------------+-------+--------------+-------+----------+----------+
|2             |1      |2             |59.762 |1752061984|2025-07-09|
|3             |1      |3             |518.72 |1752061984|2025-07-09|
|9             |2      |3             |515.393|1752061984|2025-07-09|
|16            |3      |4             |6.025  |1752061984|2025-07-09|
|29            |5      |5             |59.372 |1752061984|2025-07-09|
|32            |6      |2             |59.604 |1752061984|2025-07-09|
|36            |6      |6             |754.29 |1752061984|2025-07-09|
|40            |7      |4             |6.06   |1752061984|2025-07-09|
|41            |7      |5             |60.377 |1752061984|2025-07-09|
|49            |9      |1             |21.584 |1752061984|2025-

26/07/09 11:56:39 WARN DAGScheduler: Failed to cancel job group fc2b18dc-a50c-47b1-8b1d-d8ec3720ecf8. Cannot find active jobs for it.
26/07/09 11:56:39 WARN DAGScheduler: Failed to cancel job group fc2b18dc-a50c-47b1-8b1d-d8ec3720ecf8. Cannot find active jobs for it.


In [13]:
def write_stream(df, config):
    output_path = (
        f"s3a://{config['staging_bucket']}/raw/kafka/{config['kafka_topic']}/"
    )

    checkpoint_path = (
        f"s3a://{config['staging_bucket']}/_checkpoints/kafka/{config['kafka_topic']}/"
    )

    return (
        df.writeStream
        .format("parquet")
        .outputMode("append")
        .option("path", output_path)
        .option("checkpointLocation", checkpoint_path)
        .partitionBy("event_date")
        .trigger(processingTime=config["trigger_interval"])
        .start()
    )

In [ ]:
query = write_stream(parsed_df, config)
query.awaitTermination()

26/07/09 11:56:39 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
26/07/09 11:56:39 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/07/09 13:57:51 ERROR TaskSchedulerImpl: Lost executor 1 on 172.19.0.7: Worker shutting down
26/07/09 13:57:51 ERROR TaskSchedulerImpl: Lost executor 0 on 172.19.0.8: Worker shutting down
26/07/09 13:57:52 WARN StandaloneAppClient$ClientEndpoint: Connection to urbangreen-spark-master:7077 failed; waiting for master to reconnect...
26/07/09 13:57:52 WARN StandaloneSchedulerBackend: Disconnected from Spark cluster! Waiting for reconnection...
26/07/09 13:57:52 WARN StandaloneAppClient$ClientEndpoin

In [ ]:
spark.stop()